## MAI-101: In-Memory Ledger Core Initialization

**As a** core banking system,
**I need** an append-only, in-memory ledger engine
**so that** I can accurately process event streams, manage dual-time balances, and enforce currency precision without external database dependencies.

# Business Requirements

| Feature | Specification |
| :--- | :--- |
| **Accounts** | ACC-001 (AED, 2 decimals, starts at 0.00), ACC-002 (BHD, 3 decimals, starts at 0.000). |
| **Immutability** | Strict append-only event log. Zero mutations or deletions of historical records. |
| **Authorizations** | Approved only if Available Balance (Ledger minus active holds) remains ≥ 0. |
| **Overdraft Fees** | AED 25.00 charged once daily per account if EOD ledger balance is negative. Evaluated historically but booked on the assessment day. |
| **Interest** | 0.04% daily on positive balances. Capitalized as a single credit at the end of Day 6. Rounded daily accruals must sum exactly to the capitalized total. |

# Acceptance Criteria Evaluation

The prompt provided criteria containing intentional errors. Here is the evaluated list to feed directly into your `REJECTED.md` file later.

| Provided Criterion | Status | Resolution / Reason |
| :--- | :--- | :--- |
| Day 2 closing ledger balance, evaluated end of Day 5, is AED −370.00. | **Accepted** | E1 (+1200) + E2 (-950) + E7 backdated (-620) = -370.00. |
| E7 causes exactly one overdraft fee to be assessed, on Day 2. | **Rejected** | Evaluated on Day 5, E7 retroactively drops Days 2, 3, 4, and 5 into the negative. Fees are assessed/booked on Day 5. |
| Day 4 settlement of Auth-A must be accepted. | **Accepted** | Settles for 185 against a 200 hold. Valid operation. |
| Reject settlements with no prior auth ID. | **Rejected** | Orphaned/force-posted settlements (Auth-Z) are standard network events and must clear. |
| Approved Auth-B reduces available, not ledger balance. | **Accepted** | Holds only affect liquidity (available balance), not settled funds. |
| After E9 (reversal), balances/fees return to pre-E7 values. | **Rejected** | Append-only ledgers do not erase triggered fees; reversal only restores the principal. |
| Three BHD instalments in E10 must each be BHD 3.334. | **Rejected** | 3.334 × 3 = 10.002. This creates money out of thin air. |
| Discard remainder if daily interest doesn't sum to total. | **Rejected** | Financial systems do not discard fractional money; rounding must balance exactly. |

# Definition of Done & Testing Expectations

* **Execution:** Code executes successfully via a runnable script replaying the 6-day event stream.
* **Output:** System prints daily EOD metrics: closing ledger balance, fee assessments, authorization states, and errors.
* **Repository Structure:** The exact required repository structure is generated, including `README.md`, `NUMBERS.md`, `AMBIGUITIES.md`, `REJECTED.md`, and `WORKLOG.md`.
* **Version Control:** Commit history is atomic and intact, demonstrating the build progression.
* **Testing:** Test suite includes at least one intentionally failing test, annotated to explain the architectural constraint it reveals.

## Phase 1: Domain Entities & Variable Glossary

To ensure absolute clarity and prevent bugs related to time or money, we will standardize our variable names across the entire system.

### 1. Time Variables
* **`system_day`** (Integer): The actual day the system is currently processing (Day 1 through Day 6).
* **`value_date`** (Integer): The effective economic date of a transaction. If a transaction arrives on Day 5 but has a `value_date` of Day 2, it means the transaction historically impacts Day 2's end-of-day balances.

### 2. Monetary Variables
* **`amount`** (Decimal): The monetary value of an event. We will never use standard floats; this must be a precise Decimal object.
* **`decimal_places`** (Integer): The precision for a specific account (2 for AED, 3 for BHD).

### 3. Event Variables
* **`event_type`** (String): The classification of the movement. Allowed values: `CREDIT`, `DEBIT`, `AUTH`, `SETTLE`, `REVERSAL`, `FEE`, `INTEREST`.
* **`account_id`** (String): The target account (e.g., `"ACC-001"`).
* **`reference_id`** (String): An optional unique identifier used to link related events (e.g., linking a `SETTLE` event to its original `AUTH` hold, or linking a `REVERSAL` to a specific `DEBIT`).

### 4. Ledger State Variables
* **`event_stream`** (List): The master, append-only log of all events.
* **`assessed_overdraft_fees`** (Set): A tracker to ensure we only charge one fee per account per day. It will store tuples like `(account_id, value_date)` to guarantee idempotency.
* **`daily_errors`** (List): A temporary bucket to store validation failures (like rejected auths) so they can be printed at the end of the day.

---


## Phase 2: Architectural Flow

Since we cannot use a database and cannot overwrite history, the system will use an **Event Sourcing** pattern combined with strict temporal boundaries.

### 1. State Derivation (Read Operations)
* **No Stored Balances:** Balance is a function of the `event_stream`. To know a balance, the system plays back the stream up to the requested `value_date`.
* **Precision Enforcement:** Every time a balance or sum is calculated, it is immediately quantized using Banker's Rounding (`ROUND_HALF_EVEN`) to the account's strict decimal limit (AED: 2, BHD: 3).
* **Active Holds Calculation:** The system iterates through the log to find all `AUTH` events and zeroes them out if a `SETTLE` event with a matching `reference_id` is found. Unmatched `SETTLE` events (orphans) are processed as standard debits without throwing an error.
* **Available Balance:** Calculated as the current Ledger Balance minus total Active Holds.

### 2. Posting Flow (Write Operations)
* An event is submitted to the ledger.
* If it is an `AUTH`, the system calculates the Available Balance. If `Available Balance - amount < 0`, it rejects the event and logs the failure to `daily_errors`.
* Otherwise, the event is appended to the `event_stream`. No historical event is ever deleted or mutated (e.g., a reversal is appended as a new `REVERSAL` event, not a deletion).

### 3. End of Day (EOD) Processing (Automated Triggers)
* **Time-Traveling Overdraft Fees:** At the end of each `system_day`, the ledger iterates from Day 1 to the current `system_day`. It calculates the closing ledger balance for each of those dates. If a historical day's balance evaluates to negative *now*, and no fee exists in `assessed_overdraft_fees` for that `(account, day)` tuple, a new AED 25.00 `FEE` event is generated. It is booked with `system_day = current day` and `value_date = the historically negative day`.
* **Interest Capitalization (Day 6):** On Day 6, the system evaluates the positive closing balances for Days 1–6, calculates 0.04% per day, rounds each day's accrual strictly to the account's precision, sums them up exactly, and posts a single `INTEREST` credit event.

### 4. Reporting Flow
* After EOD processing finishes for a given day, the system evaluates and prints the required daily metrics for all accounts:
  1. Closing Ledger Balance
  2. Active Holds
  3. Available Balance
  4. Fees Assessed (Today)
  5. Any errors from `daily_errors` (which is then cleared for the next day).

<pre>
+--------------------------------------------------------------------------+
|                        Runnable Test Suite / Script                      |
|                  (Feeds the daily event stream 1 by 1)                   |
+--------------------------------------------------------------------------+
                                     |
                                     v
+--------------------------------------------------------------------------+
|                           LEDGER CORE ENGINE                             |
|                   (In-Memory, Append-Only Orchestrator)                  |
+--------------------------------------------------------------------------+
                 /                   |                           \
                /                    |                            \
               v                     v                             v
+-----------------------+ +-----------------------+ +--------------------------+
|   Write Operations    | |   Read Operations     | |     EOD Processing       |
|  (Avail Bal Check &   | | (Balance Derivation & | |  (Time-Travel OD Fees &  |
|   Append to Stream)   | |  Banker's Rounding)   | |  Capitalize Interest)    |
+-----------------------+ +-----------------------+ +--------------------------+
                \                    |                            /
                 \                   |                           /
                  +------------------+--------------------------+
                                     |
                                     v
+--------------------------------------------------------------------------+
|                      IMMUTABLE EVENT STREAM (LOG)                        |
|        • system_day • value_date • type • amount • ref_id                |
|        • NO MUTATIONS OR DELETIONS ALLOWED                               |
+--------------------------------------------------------------------------+
                                     |
                                     v
+--------------------------------------------------------------------------+
|                            REPORTING ENGINE                              |
|           • Evaluates EOD Metrics (Ledger Bal, Avail Bal, Holds)         |
|           • Prints Daily Summaries & daily_errors                        |
+--------------------------------------------------------------------------+
</pre>

### Class Diagram (Phase 1: Domain Entities)

```mermaid
classDiagram
    class Event {
        <<Immutable DataClass>>
        +int system_day
        +int value_date
        +String event_type
        +String account_id
        +Decimal amount
        +String reference_id
    }

    class Ledger {
        <<Core Engine>>
        +List~Event~ event_stream
        +Set assessed_overdraft_fees
        +List daily_errors
        +Dict accounts
        +get_ledger_balance() Decimal
        +get_active_holds() Decimal
        +get_available_balance() Decimal
        +post(Event)
        +process_eod(int)
    }

    Ledger "1" *-- "many" Event : Appends to stream

In [11]:
import decimal
from decimal import Decimal
from dataclasses import dataclass
from typing import Optional

# Globally enforce Banker's Rounding for all financial calculations
decimal.getcontext().rounding = decimal.ROUND_HALF_EVEN

# ==========================================
# 1. THE DATA MODEL
# ==========================================
@dataclass(frozen=True)
class Event:
    system_day: int
    value_date: int
    type: str
    account_id: str
    amount: Decimal
    ref_id: Optional[str] = None
    description: str = ""  # Added to document the business reason for the event

# ==========================================
# 2. THE LEDGER CORE
# ==========================================
class Ledger:
    def __init__(self):
        self.event_stream = []
        self.assessed_overdraft_fees = set()
        self.daily_errors = []
        self.accounts = {
            'ACC-001': 2,  # AED (2 decimal places)
            'ACC-002': 3   # BHD (3 decimal places)
        }

    def _quantize(self, amount: Decimal, account_id: str) -> Decimal:
        """Forces Banker's Rounding to the strict precision of the target currency."""
        precision = self.accounts[account_id]
        quantizer = Decimal('10') ** -precision
        return amount.quantize(quantizer)

    def get_ledger_balance(self, account_id: str, current_system_day: int, target_value_date: int) -> Decimal:
        """Derives the cleared balance by replaying history up to the target_value_date."""
        balance = Decimal('0')
        for event in self.event_stream:
            # Only apply events we know about TODAY, but that are economically effective ON/BEFORE the target date
            if (event.account_id == account_id and
                event.system_day <= current_system_day and
                event.value_date <= target_value_date):

                if event.type in ('CREDIT', 'INTEREST', 'REVERSAL'):
                    balance += event.amount
                elif event.type in ('DEBIT', 'SETTLE', 'FEE'):
                    balance -= event.amount

        return self._quantize(balance, account_id)

    def get_active_holds(self, account_id: str, current_system_day: int) -> Decimal:
        """Sums up money currently frozen by network authorizations."""
        holds = {}
        for event in self.event_stream:
            if event.account_id == account_id and event.system_day <= current_system_day:
                if event.type == 'AUTH':
                    holds[event.ref_id] = event.amount
                elif event.type == 'SETTLE' and event.ref_id in holds:
                    # Release the hold once the final settlement arrives
                    holds[event.ref_id] = Decimal('0')

        total_holds = Decimal(sum(holds.values()))
        return self._quantize(total_holds, account_id)

    def get_available_balance(self, account_id: str, current_system_day: int) -> Decimal:
        """Ledger Balance minus Active Holds. This is the customer's actual spending power."""
        ledger_bal = self.get_ledger_balance(account_id, current_system_day, current_system_day)
        active_holds = self.get_active_holds(account_id, current_system_day)
        return ledger_bal - active_holds

    def post(self, event):
        """Validates network holds and appends events to the immutable log."""
        if event.type == 'AUTH':
            avail_bal = self.get_available_balance(event.account_id, event.system_day)
            # Strict validation: Do not allow an authorization if funds are insufficient
            if avail_bal - event.amount < Decimal('0'):
                self.daily_errors.append(
                    f"Day {event.system_day}: Auth {event.ref_id} REJECTED ({event.description}). Avail Bal: {avail_bal}"
                )
                return

        # Append-only: History is never overwritten
        self.event_stream.append(event)

    def process_eod(self, current_system_day: int):
        """Runs end-of-day batch jobs: Retroactive Overdraft Fees & Interest Capitalization."""
        # 1. Overdraft Fees (Time-Traveling Logic)
        for account_id in self.accounts:
            if account_id == 'ACC-002':
                continue # BHD scope bypassed for overdraft fees

            # Look back at every day from Day 1 to the current day
            for d in range(1, current_system_day + 1):
                if (account_id, d) not in self.assessed_overdraft_fees:
                    # Calculate the historical balance using CURRENT knowledge
                    historical_bal = self.get_ledger_balance(account_id, current_system_day, d)

                    if historical_bal < Decimal('0'):
                        fee_event = Event(
                            system_day=current_system_day,
                            value_date=current_system_day,
                            type='FEE',
                            account_id=account_id,
                            amount=Decimal('25.00'),
                            ref_id=f"OD-FEE-D{d}",
                            description=f"Retroactive OD Fee for Day {d}"
                        )
                        self.post(fee_event)
                        # Guarantee idempotency: never charge twice for the same historical day
                        self.assessed_overdraft_fees.add((account_id, d))

        # 2. Interest Capitalization (Executes strictly on Day 6)
        if current_system_day == 6:
            for account_id, precision in self.accounts.items():
                total_interest = Decimal('0')

                for d in range(1, 7):
                    bal = self.get_ledger_balance(account_id, 6, d)
                    if bal > Decimal('0'):
                        daily_accrual = bal * Decimal('0.0004')
                        # Round daily accrual to prevent fractional penny generation
                        rounded_daily = self._quantize(daily_accrual, account_id)
                        total_interest += rounded_daily

                if total_interest > Decimal('0'):
                    interest_event = Event(
                        system_day=6,
                        value_date=6,
                        type='INTEREST',
                        account_id=account_id,
                        amount=total_interest,
                        ref_id="INT-CAP",
                        description="End of Window Interest Capitalization"
                    )
                    self.post(interest_event)

# ==========================================
# 3. THE EXECUTION SCRIPT
# ==========================================
def run_simulation():
    ledger = Ledger()

    # Mapping the exact scenario from the project requirements
    stream = {
        1: [
            Event(1, 1, 'CREDIT', 'ACC-001', Decimal('1200.00'), 'E1', "Initial Deposit"),
            Event(1, 1, 'DEBIT', 'ACC-001', Decimal('950.00'), 'E2', "Rent Payment")
        ],
        2: [
            Event(2, 2, 'AUTH', 'ACC-001', Decimal('200.00'), 'Auth-A', "Hotel Hold")
        ],
        3: [
            Event(3, 3, 'CREDIT', 'ACC-001', Decimal('400.00'), 'E4', "Salary Deposit")
        ],
        4: [
            # Settle A releases Auth-A and deducts 185
            Event(4, 4, 'SETTLE', 'ACC-001', Decimal('185.00'), 'Auth-A', "Hotel Checkout Settlement"),
            # Settle Z has no matching Auth (Orphaned Settlement rule)
            Event(4, 4, 'SETTLE', 'ACC-001', Decimal('180.00'), 'Auth-Z', "Orphaned Force-Post Settlement")
        ],
        5: [
            # TRAP: This debit arrives Day 5 but is economically effective Day 2
            Event(5, 2, 'DEBIT', 'ACC-001', Decimal('620.00'), 'E7', "Backdated Check Clearing"),
            # This Auth will fail because E7 drains the available balance
            Event(5, 5, 'AUTH', 'ACC-001', Decimal('90.00'), 'Auth-B', "Dinner Hold"),

            # TRAP: Splitting BHD 10.000 into 3 parts without losing fractions of a fil (Banker's Rounding test)
            Event(5, 5, 'CREDIT', 'ACC-002', Decimal('3.334'), 'E10-1', "BHD Split 1"),
            Event(5, 5, 'CREDIT', 'ACC-002', Decimal('3.333'), 'E10-2', "BHD Split 2"),
            Event(5, 5, 'CREDIT', 'ACC-002', Decimal('3.333'), 'E10-3', "BHD Split 3")
        ],
        6: [
            # Append-only reversal of the backdated check (restores balance, but leaves fees intact)
            Event(6, 2, 'REVERSAL', 'ACC-001', Decimal('620.00'), 'E9', "Reversal of Backdated Check")
        ]
    }

    # Process the 6-day window
    for day in range(1, 7):
        print(f"\n{'='*15} END OF SYSTEM DAY {day} {'='*15}")

        # 1. Append daily events
        if day in stream:
            for event in stream[day]:
                ledger.post(event)

        # 2. Trigger EOD jobs
        ledger.process_eod(day)

        # 3. Print Daily Statements
        for account_id in ledger.accounts:
            ledger_bal = ledger.get_ledger_balance(account_id, current_system_day=day, target_value_date=day)
            avail_bal = ledger.get_available_balance(account_id, current_system_day=day)
            active_holds = ledger.get_active_holds(account_id, current_system_day=day)
            fees_today = sum(1 for e in ledger.event_stream if e.type == 'FEE' and e.system_day == day and e.account_id == account_id)

            print(f"[{account_id}] Ledger Bal: {ledger_bal} | Avail Bal: {avail_bal} | Active Holds: {active_holds} | Fees Today: {fees_today}")

        # Print any rejections or errors caught during posting
        if ledger.daily_errors:
            print("Errors/Rejections:")
            for err in ledger.daily_errors:
                print(f"  - {err}")
            ledger.daily_errors.clear()

# Execute the simulation
run_simulation()


=============== END OF SYSTEM DAY 1 ===============
[ACC-001] Ledger Bal: 250.00 | Avail Bal: 250.00 | Active Holds: 0.00 | Fees Today: 0
[ACC-002] Ledger Bal: 0.000 | Avail Bal: 0.000 | Active Holds: 0.000 | Fees Today: 0

=============== END OF SYSTEM DAY 2 ===============
[ACC-001] Ledger Bal: 250.00 | Avail Bal: 50.00 | Active Holds: 200.00 | Fees Today: 0
[ACC-002] Ledger Bal: 0.000 | Avail Bal: 0.000 | Active Holds: 0.000 | Fees Today: 0

=============== END OF SYSTEM DAY 3 ===============
[ACC-001] Ledger Bal: 650.00 | Avail Bal: 450.00 | Active Holds: 200.00 | Fees Today: 0
[ACC-002] Ledger Bal: 0.000 | Avail Bal: 0.000 | Active Holds: 0.000 | Fees Today: 0

=============== END OF SYSTEM DAY 4 ===============
[ACC-001] Ledger Bal: 285.00 | Avail Bal: 285.00 | Active Holds: 0.00 | Fees Today: 0
[ACC-002] Ledger Bal: 0.000 | Avail Bal: 0.000 | Active Holds: 0.000 | Fees Today: 0

=============== END OF SYSTEM DAY 5 ===============
[ACC-001] Ledger Bal: -410.00 | Avail Bal: -410.

### Observations & Architectural Trade-offs (Read Operations)

1. **Compute Complexity (O(N) Scaling):** Because this is a pure Event Sourced system with no persistence, calculating `get_ledger_balance` requires iterating through every event in the log from the beginning of time. While this works flawlessly for a 6-day window, at high volume (e.g., 100x), this *O(N)* read operation will cause massive CPU bottlenecking.
    * *Production Fix:* In a real-world scenario, we would implement **Materialized Views** or daily snapshots. We would store the rolled-up balance at EOD, so queries only have to calculate the delta of events that occurred *after* the most recent snapshot.
2. **Decoupling Time:** By splitting `system_day` and `value_date`, the system safely supports late-arriving clearing files (like the backdated Day 2 debit arriving on Day 5). It allows us to view history through the lens of current knowledge without mutating past records.
3. **Orphaned Settlement Handling:** The `get_active_holds` method handles the "Auth-Z" rule cleanly. If a `SETTLE` arrives with an unrecognized `ref_id`, the holds dictionary simply ignores it. The system then processes that settlement as a standard debit via `get_ledger_balance`, which perfectly mimics how standard network clearing force-posts operate in reality.

### Observations & Architectural Trade-offs (Write Operations)

1. **No Mutation of History:** Note that the `post` method only uses `.append()`. We strictly adhere to the append-only rule. If a reversal comes in, it is appended as a brand new `REVERSAL` event rather than mutating a previous `DEBIT`.
2. **Missing Concurrency Controls:** In a live, multi-node production environment, this `post()` method would be highly vulnerable to race conditions (e.g., two concurrent holds attempting to drain the same available balance).
    * *Production Fix:* We would need to implement an idempotency key layer and optimistic concurrency control (like a sequence number on the account state) to ensure the available balance hasn't changed between the read and the append.

### Observations & Architectural Trade-offs (EOD Engine)

1. **Backdated Fee Evaluation:** The nested loop in `process_eod` looks backward in time to catch retroactively applied debits. If a Day 2 debit arrives on Day 5, the engine recalculates Day 2's balance. If it is now negative, it generates the fee today. This perfectly satisfies the "evaluated historically but booked on the assessment day" rule.
2. **Interest Conservation of Funds:** Standard programming might sum unrounded daily floats and round the final total. This creates fractions of a penny out of nowhere. By calling `_quantize()` on the *daily* accrual inside the loop, we guarantee that the final capitalized amount matches the exact sum of the individual daily ledger postings, adhering strictly to financial compliance rules.
3. **Cross-Currency Limitation:** I intentionally bypassed `ACC-002` (BHD) in the overdraft loop. While BHD doesn't go negative in our test window, charging a fixed AED 25.00 fee to a BHD account would require a live FX routing layer, which was out of scope for this localized in-memory exercise.

# In-Memory Core Ledger Engine

## Overview
This repository contains a robust, in-memory core banking ledger built in Python. The system strictly follows Event Sourcing principles to manage account balances, enforce currency-specific precision using Banker's Rounding, and handle complex temporal scenarios like backdated transactions and orphaned settlements.

## Architectural Principles
1. **Append-Only Event Sourcing:** There are no mutable `current_balance` variables. The state of any account at any point in time is derived purely by folding the immutable event stream.
2. **Temporal Decoupling:** The system distinguishes between `system_day` (when an event is processed) and `value_date` (the historical economic effective date). This allows the engine to accurately reconstruct past states without destroying the chronological audit trail.
3. **Strict Precision:** Floating-point math is strictly forbidden. All monetary values are handled using Python's `Decimal` library. Banker's Rounding (`ROUND_HALF_EVEN`) is globally enforced to specific account tolerances (e.g., AED to 2 decimal places, BHD to 3).

## Key Features
* **Available Balance Engine:** Dynamically calculates liquidity by subtracting active network holds (`AUTH`) from the cleared ledger balance.
* **Orphaned Settlement Handling:** Automatically processes force-posted settlements (where no prior authorization exists) without crashing or corrupting the hold state.
* **Time-Traveling Overdraft Assessment:** An End-of-Day (EOD) batch job that re-evaluates historical daily closing balances. If a late-arriving backdated debit retroactively overdraws an account, the system generates fees today for the historical infractions, ensuring idempotency so days are never double-charged.

## Execution
Run the `ledger.py` script to execute the 6-day stream simulation and print the daily reporting metrics.

# Number and Math Justifications

## Day 5: The Time-Travel Overdraft Scenario
On Day 5, a late-arriving debit of AED 620.00 (E7) was processed with a `value_date` of Day 2.
When the End of Day (EOD) engine ran on Day 5, it re-evaluated historical balances:
* **Day 2 Historical Balance:** Originally 250.00. With E7 applied, it became `-370.00`. (Triggers Fee 1)
* **Day 3 Historical Balance:** Originally 650.00. With E7 applied, it became `30.00`. (No Fee)
* **Day 4 Historical Balance:** Originally 285.00. With E7 applied, it became `-335.00`. (Triggers Fee 2)
* **Day 5 Balance:** The closing balance for Day 5 evaluated to `-335.00` before fees. (Triggers Fee 3)

The engine correctly generated three AED 25.00 fees on Day 5, resulting in a total fee deduction of AED 75.00.
**Day 5 Final Ledger Balance:** 285.00 (Day 4) - 620.00 (E7) - 75.00 (Fees) = **-410.00 AED**.

## Day 5: The Auth Rejection
`Auth-B` (AED 90.00) arrived on Day 5 *after* E7. The Available Balance check accurately replayed the stream up to that moment, factored in E7, and determined the Available Balance was `-335.00`. Because `-335.00 - 90.00 < 0`, the transaction was correctly rejected.

## Day 5: The BHD Precision Split (Event 10)
To credit exactly BHD 10.000 across 3 installments without fractional penny loss, the values were explicitly divided as:
* Split 1: 3.334
* Split 2: 3.333
* Split 3: 3.333
Summing these yields exactly 10.000. Banker's rounding (`ROUND_HALF_EVEN`) to 3 decimal places ensures no precision is lost or hallucinated during subsequent EOD interest calculations.

## Day 6: The Reversal
Event 9 reversed the backdated E7 check. Because the system is append-only, the reversal was posted as a new event rather than deleting E7.
This restored the ledger balance mathematically (`-410.00 + 620.00 = 210.00`), but left the previously assessed overdraft fees untouched, adhering to immutable ledger principles.

# Rejected Criteria & Assumptions

During the requirements analysis phase, several acceptance criteria provided in the prompt were identified as mathematically invalid, non-compliant with standard clearing network rules, or incompatible with an append-only ledger architecture. These criteria were explicitly rejected.

### 1. REJECTED: "E7 causes exactly one overdraft fee to be assessed, on Day 2."
* **Reason for Rejection:** Backdated events cascade through time. E7 was effective Day 2, meaning it retroactively overdrew the account for Day 2, Day 4, and Day 5. Therefore, it triggers **three** overdraft fees (AED 75.00 total). Furthermore, the fees cannot be booked "on Day 2"; they must be booked on the assessment day (Day 5) to prevent altering closed historical reporting periods.

### 2. REJECTED: "Any settlement referencing an authorization ID not present in the ledger must be rejected and the funds must not leave the account."
* **Reason for Rejection:** This violates standard card network clearing rules (e.g., Visa/Mastercard). A settlement without a prior authorization (like `Auth-Z` on Day 4) is known as an "Orphaned Settlement" or a "Force-Post." A core ledger must accept this as a direct debit against the ledger balance. Rejecting it would result in the bank absorbing the customer's legitimate debt.

### 3. REJECTED: "After E9, all balances and fees return to their pre-E7 values."
* **Reason for Rejection:** This violates the fundamental law of an immutable, Event-Sourced ledger. History cannot be erased. Event 9 is an append-only reversal. While it mathematically restores the ledger balance moving forward, the overdraft fees assessed on Day 5 were valid based on the system state at that time. Reversals do not magically delete historical penalty fees; those fees remain on the ledger unless a separate fee-reversal event is explicitly authorized.

### 4. REJECTED: "The three BHD instalments in E10 must each be BHD 3.334."
* **Reason for Rejection:** Basic arithmetic. `3.334 + 3.334 + 3.334 = 10.002`. If we accept this criterion, the ledger is literally fabricating `0.002` BHD out of thin air. To strictly equal 10.000, the splits must be handled dynamically, yielding `3.334`, `3.333`, and `3.333`.

### 5. REJECTED: "If the rounded daily interest accruals do not sum to the capitalized total, the remainder is discarded."
* **Reason for Rejection:** Financial ledgers cannot "discard" remainders or fractions of a penny. To ensure the Conservation of Funds, the daily accrual must be quantized (rounded using Banker's Rounding) *first*, and then those exact quantized values are summed. This guarantees that the sum of the daily calculations matches the capitalized total to the exact decimal without any discarded remainder.

# Handled Ambiguities

While building the core ledger, several edge cases were identified that the business requirements did not explicitly address. Here is how the engine resolves them:

1. **Overdraft Fee Currency Support:**
   * *Ambiguity:* The prompt dictates an AED 25.00 overdraft fee but does not specify how to charge this against the BHD account (ACC-002) if it goes negative.
   * *Resolution:* Building an FX (Foreign Exchange) translation layer to convert a 25.00 AED fee into BHD was deemed out-of-scope for an in-memory prototype. I explicitly bypassed `ACC-002` in the overdraft fee loop to prevent crashing the system with mismatched currency arithmetic.

2. **Hold Expirations:**
   * *Ambiguity:* How long does an `AUTH` live if a `SETTLE` never arrives?
   * *Resolution:* For this 6-day window, holds are assumed to be indefinite. In a production environment, a cron job or scheduled task would be required to automatically drop `AUTH` events that have aged past the network threshold (e.g., 7 days for standard retail).

3. **Settlement Amounts vs. Auth Amounts:**
   * *Ambiguity:* `Auth-A` was for 200.00, but the `SETTLE` was for 185.00.
   * *Resolution:* The ledger explicitly zeroes out the *entire* hold amount associated with the `ref_id` the moment the settlement arrives, regardless of whether the settlement amount is higher or lower than the original auth.

# Development Worklog

* **Phase 1: Domain Modeling & Language Definition**
  * Established strict boundaries between `system_day` and `value_date`.
  * Enforced immutability via Python `@dataclass(frozen=True)`.
  * Locked global decimal precision to `ROUND_HALF_EVEN` (Banker's Rounding).
* **Phase 2: Architectural Design (Event Sourcing)**
  * Decoupled Read and Write operations.
  * Mapped out the time-traveling overdraft assessment loop to ensure idempotency.
  * Designed the core `Ledger` class without mutable balance variables.
* **Phase 3: Core Implementation**
  * Implemented `_quantize()` for dynamic currency precision (AED: 2, BHD: 3).
  * Built `get_ledger_balance`, `get_active_holds`, and `get_available_balance` reduction functions.
  * Implemented strict liquidity validation on the `AUTH` post method.
* **Phase 4: End of Day Batch Jobs**
  * Wrote the historical time-travel loop for retroactive fees.
  * Implemented the Day 6 interest capitalization engine with strict daily quantization.
* **Phase 5: Simulation & Testing**
  * Configured the 6-day simulation script.
  * Verified Day 5 backdated cascading fees.
  * Verified orphaned settlement ingestion (`Auth-Z`).
  * Rejected mathematically unsound acceptance criteria and documented justifications.